In [1]:
!pip install -q open_clip_torch

import sys, shutil, torch
from pathlib import Path

SRC = Path("/kaggle/input/datasets/aaryaupi/derm-audit-cod") 
# /kaggle/input/datasets/aaryaupi/derm-audit-cod
for f in ["run_derm7pt.py", "derm_audit_core.py", "protocol.py", "clinical_prompts.json"]:
    shutil.copy(SRC / f, f"/kaggle/working/{f}")
sys.path.insert(0, "/kaggle/working")

print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 12.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
GPU: True Tesla T4


In [2]:
!cd /kaggle/working && python derm_audit_core.py

SELFTEST: fake encoder, planted signal on 2 of 7 concepts.

tests=42  alpha=1.19e-03  smallest observable p=0.0010

=== ref ===
              concept  n_pos  auroc  null_med  null_p95  beaten_by      p sig
      pigment_network    111  0.521     0.496     0.536        162 0.1628    
    blue_whitish_veil    129  0.899     0.560     0.656          0 0.0010 YES
  vascular_structures    128  0.479     0.494     0.530        745 0.7453    
         pigmentation    114  0.484     0.511     0.549        842 0.8422    
              streaks    112  0.911     0.519     0.625          0 0.0010 YES
    dots_and_globules    119  0.501     0.480     0.525        228 0.2288    
regression_structures    118  0.537     0.506     0.551        125 0.1259    
recovered: ['blue_whitish_veil', 'streaks']

=== antonym ===
              concept  n_pos  auroc  null_med  null_p95  beaten_by      p sig
      pigment_network    111  0.489     0.501     0.559        611 0.6114    
    blue_whitish_veil    129  0

In [3]:
import run_derm7pt as R
R.DIR = Path("/kaggle/input/datasets/menakamohanakumar/derm7pt/release_v0")
df, concepts, mel, splits, ceil = R.run_all(stage="checks")

meta.csv: 1011 rows
dropping 9 flattened-modality rows -> n=1002
  the dropped rows:
     case_num   case_id                    diagnosis          clinic
564       565       NaN                      lentigo  Fhl/Fhl018.jpg
799       800  1b514950  melanoma (more than 1.5 mm)  NBL/NBL049.JPG
823       824       NaN          melanoma metastasis  NBL/NBL041.JPG
826       827       NaN          melanoma metastasis  NBL/Nbl042.jpg
832       833       NaN                    melanosis  Fgl/Fgl059.jpg
833       834       NaN                    melanosis  Fhl/Fhl002.jpg
838       839       NaN                    melanosis  Fhl/Fhl012.jpg
843       844       NaN                miscellaneous  NEL/Nel050.jpg
850       851       NaN                miscellaneous  NEL/Nel051.jpg

=== DATA CHECKS ===

positives vs published (all rows kept in the published count, we drop 9):
  pigment_network          ours= 229  published= 230  ok
  blue_whitish_veil        ours= 192  published= 195  ok
  vascular_stru

In [4]:
R.run_all(backbone="clip", stage="all", eval_split="test", n_draws=1000)

meta.csv: 1011 rows
dropping 9 flattened-modality rows -> n=1002
  the dropped rows:
     case_num   case_id                    diagnosis          clinic
564       565       NaN                      lentigo  Fhl/Fhl018.jpg
799       800  1b514950  melanoma (more than 1.5 mm)  NBL/NBL049.JPG
823       824       NaN          melanoma metastasis  NBL/NBL041.JPG
826       827       NaN          melanoma metastasis  NBL/Nbl042.jpg
832       833       NaN                    melanosis  Fgl/Fgl059.jpg
833       834       NaN                    melanosis  Fhl/Fhl002.jpg
838       839       NaN                    melanosis  Fhl/Fhl012.jpg
843       844       NaN                miscellaneous  NEL/Nel050.jpg
850       851       NaN                miscellaneous  NEL/Nel051.jpg

=== DATA CHECKS ===

positives vs published (all rows kept in the published count, we drop 9):
  pigment_network          ours= 229  published= 230  ok
  blue_whitish_veil        ours= 192  published= 195  ok
  vascular_stru

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

loaded open_clip ViT-L-14-quickgelu / openai
  sanity: dim=768, cos range [0.153, 0.284], red-vs-blue text separation +0.067
  896/1002
=== clip :: ref_noens ===
     cell               concept  n_pos  auroc  null_med  null_p95  beaten_by      p sig  floor_p95  headroom
ref_noens       pigment_network     93  0.592     0.516     0.567          8 0.0090          0.567     0.025
ref_noens     blue_whitish_veil     74  0.529     0.442     0.528         43 0.0440          0.528     0.001
ref_noens   vascular_structures     30  0.568     0.547     0.631        351 0.3516          0.631    -0.063
ref_noens          pigmentation    124  0.546     0.473     0.529         12 0.0130          0.529     0.017
ref_noens               streaks     94  0.501     0.477     0.532        233 0.2338          0.532    -0.031
ref_noens     dots_and_globules    176  0.508     0.483     0.532        207 0.2078          0.532    -0.024
ref_noens regression_structures    105  0.590     0.558     0.628        26

(      case_num             diagnosis  seven_point_score pigment_network  \
 0            1  basal cell carcinoma                  0          absent   
 1            2  basal cell carcinoma                  1          absent   
 2            3  basal cell carcinoma                  1          absent   
 3            4  basal cell carcinoma                  4          absent   
 4            5  basal cell carcinoma                  1          absent   
 ...        ...                   ...                ...             ...   
 1006      1007       vascular lesion                  0          absent   
 1007      1008       vascular lesion                  0          absent   
 1008      1009       vascular lesion                  0          absent   
 1009      1010       vascular lesion                  0          absent   
 1010      1011              melanoma                  2          absent   
 
      streaks       pigmentation regression_structures dots_and_globules  \
 0     abs

In [9]:
R.run_all(backbone="clip", stage="arms")

meta.csv: 1011 rows
dropping 9 flattened-modality rows -> n=1002
  the dropped rows:
     case_num   case_id                    diagnosis          clinic
564       565       NaN                      lentigo  Fhl/Fhl018.jpg
799       800  1b514950  melanoma (more than 1.5 mm)  NBL/NBL049.JPG
823       824       NaN          melanoma metastasis  NBL/NBL041.JPG
826       827       NaN          melanoma metastasis  NBL/Nbl042.jpg
832       833       NaN                    melanosis  Fgl/Fgl059.jpg
833       834       NaN                    melanosis  Fhl/Fhl002.jpg
838       839       NaN                    melanosis  Fhl/Fhl012.jpg
843       844       NaN                miscellaneous  NEL/Nel050.jpg
850       851       NaN                miscellaneous  NEL/Nel051.jpg

=== DATA CHECKS ===

positives vs published (all rows kept in the published count, we drop 9):
  pigment_network          ours= 229  published= 230  ok
  blue_whitish_veil        ours= 192  published= 195  ok
  vascular_stru

(      case_num             diagnosis  seven_point_score pigment_network  \
 0            1  basal cell carcinoma                  0          absent   
 1            2  basal cell carcinoma                  1          absent   
 2            3  basal cell carcinoma                  1          absent   
 3            4  basal cell carcinoma                  4          absent   
 4            5  basal cell carcinoma                  1          absent   
 ...        ...                   ...                ...             ...   
 1006      1007       vascular lesion                  0          absent   
 1007      1008       vascular lesion                  0          absent   
 1008      1009       vascular lesion                  0          absent   
 1009      1010       vascular lesion                  0          absent   
 1010      1011              melanoma                  2          absent   
 
      streaks       pigmentation regression_structures dots_and_globules  \
 0     abs

In [5]:
!pip install -q --no-deps git+https://github.com/openai/CLIP.git ftfy regex

  Preparing metadata (setup.py) ... done


In [6]:
import importlib, run_derm7pt as R
importlib.reload(R)
print("has _build_monet:", hasattr(R, "_build_monet"))
print("has monet_transform:", hasattr(R, "monet_transform"))
print("MODELS:", R.MODELS)

has _build_monet: True
has monet_transform: True
MODELS: {'clip': ('ViT-L-14-quickgelu', 'openai'), 'clip_plain': ('ViT-L-14', 'openai'), 'monet': ('hf', 'chanwkim/monet')}


In [7]:
R.run_all(backbone="monet", stage="all", n_draws=1000)

meta.csv: 1011 rows
dropping 9 flattened-modality rows -> n=1002
  the dropped rows:
     case_num   case_id                    diagnosis          clinic
564       565       NaN                      lentigo  Fhl/Fhl018.jpg
799       800  1b514950  melanoma (more than 1.5 mm)  NBL/NBL049.JPG
823       824       NaN          melanoma metastasis  NBL/NBL041.JPG
826       827       NaN          melanoma metastasis  NBL/Nbl042.jpg
832       833       NaN                    melanosis  Fgl/Fgl059.jpg
833       834       NaN                    melanosis  Fhl/Fhl002.jpg
838       839       NaN                    melanosis  Fhl/Fhl012.jpg
843       844       NaN                miscellaneous  NEL/Nel050.jpg
850       851       NaN                miscellaneous  NEL/Nel051.jpg

=== DATA CHECKS ===

positives vs published (all rows kept in the published count, we drop 9):
  pigment_network          ours= 229  published= 230  ok
  blue_whitish_veil        ours= 192  published= 195  ok
  vascular_stru

100%|████████████████████████████████████████| 890M/890M [00:06<00:00, 134MiB/s]


Downloading: "https://aimslab.cs.washington.edu/MONET/weight_clip.pt" to /root/.cache/torch/hub/checkpoints/weight_clip.pt


100%|██████████| 889M/889M [00:30<00:00, 30.9MB/s] 


MONET via openai/CLIP: 0 missing, 0 unexpected
  sanity: dim=768, cos range [0.231, 0.295], red-vs-blue text separation +0.019
  1002/1002
cached (1002, 768) -> /kaggle/working/cache/img_emb_monet.npy
tests=42  alpha=1.19e-03  smallest observable p=0.0010

null for ref ens=False (1000 draws)...

=== monet :: ref_noens ===
     cell               concept  n_pos  auroc  null_med  null_p95  beaten_by      p sig  floor_p95  headroom
ref_noens       pigment_network     93  0.602     0.468     0.543          0 0.0010 YES      0.543     0.059
ref_noens     blue_whitish_veil     74  0.704     0.508     0.626         16 0.0170          0.626     0.078
ref_noens   vascular_structures     30  0.665     0.488     0.603          1 0.0020          0.603     0.062
ref_noens          pigmentation    124  0.577     0.462     0.550         15 0.0160          0.550     0.027
ref_noens               streaks     94  0.612     0.498     0.573          6 0.0070          0.573     0.039
ref_noens     dots_and

(      case_num             diagnosis  seven_point_score pigment_network  \
 0            1  basal cell carcinoma                  0          absent   
 1            2  basal cell carcinoma                  1          absent   
 2            3  basal cell carcinoma                  1          absent   
 3            4  basal cell carcinoma                  4          absent   
 4            5  basal cell carcinoma                  1          absent   
 ...        ...                   ...                ...             ...   
 1006      1007       vascular lesion                  0          absent   
 1007      1008       vascular lesion                  0          absent   
 1008      1009       vascular lesion                  0          absent   
 1009      1010       vascular lesion                  0          absent   
 1010      1011              melanoma                  2          absent   
 
      streaks       pigmentation regression_structures dots_and_globules  \
 0     abs

In [8]:
R.run_all(backbone="monet", stage="arms")

meta.csv: 1011 rows
dropping 9 flattened-modality rows -> n=1002
  the dropped rows:
     case_num   case_id                    diagnosis          clinic
564       565       NaN                      lentigo  Fhl/Fhl018.jpg
799       800  1b514950  melanoma (more than 1.5 mm)  NBL/NBL049.JPG
823       824       NaN          melanoma metastasis  NBL/NBL041.JPG
826       827       NaN          melanoma metastasis  NBL/Nbl042.jpg
832       833       NaN                    melanosis  Fgl/Fgl059.jpg
833       834       NaN                    melanosis  Fhl/Fhl002.jpg
838       839       NaN                    melanosis  Fhl/Fhl012.jpg
843       844       NaN                miscellaneous  NEL/Nel050.jpg
850       851       NaN                miscellaneous  NEL/Nel051.jpg

=== DATA CHECKS ===

positives vs published (all rows kept in the published count, we drop 9):
  pigment_network          ours= 229  published= 230  ok
  blue_whitish_veil        ours= 192  published= 195  ok
  vascular_stru

(      case_num             diagnosis  seven_point_score pigment_network  \
 0            1  basal cell carcinoma                  0          absent   
 1            2  basal cell carcinoma                  1          absent   
 2            3  basal cell carcinoma                  1          absent   
 3            4  basal cell carcinoma                  4          absent   
 4            5  basal cell carcinoma                  1          absent   
 ...        ...                   ...                ...             ...   
 1006      1007       vascular lesion                  0          absent   
 1007      1008       vascular lesion                  0          absent   
 1008      1009       vascular lesion                  0          absent   
 1009      1010       vascular lesion                  0          absent   
 1010      1011              melanoma                  2          absent   
 
      streaks       pigmentation regression_structures dots_and_globules  \
 0     abs